# 月データ探索ツール（標準編）

公開データセットを、**自分でX軸・Y軸を選びながら**散布図で見比べる教材です。
情報Ⅰの範囲（散布図・基本統計量・相関係数）を超える内容（検定・回帰・疑似相関の検証など）は
`explore_advanced.ipynb`（発展編）にまとめてあります。

使い方：
1. 上から順にセルを実行する（Colabなら「ランタイム」→「すべてのセルを実行」）
2. 一番下に出てくるプルダウンで、データセット・X軸・Y軸・色分けを自由に選ぶ
3. まずは何も予想せず、いろいろな組み合わせを試してみる
4. 「気になる関係」が見つかったら、ワークシート（docs/worksheet.pdf）に書き出す

> 迷ったら、いちばん下の「問いのヒント」を参考にしてください。

In [1]:
# ライブラリの読み込みと実行環境の確認
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import ipywidgets as widgets
from IPython.display import display, clear_output

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB and not os.path.exists('data'):
    print("⚠️ dataフォルダが見つかりません。")
    print("Colabでこのノートブックだけを開いた場合、データファイルは一緒に来ません。")
    print("次の2行のコメントを外して実行し、リポジトリごと取得してください：")
    print("  # !git clone https://github.com/<ユーザー名>/<リポジトリ名>.git")
    print("  # %cd <リポジトリ名>/notebooks")

# 日本語フォントの設定
# matplotlibの標準フォントには日本語が含まれておらず、グラフの日本語ラベルが
# 文字化け（豆腐□）することがある。Windows/Colabどちらでも確実に表示できるよう、
# フォントファイル（Noto Sans JP）をこのリポジトリに同梱して読み込む。
_font_path = None
for _p in ('assets/NotoSansJP-Regular.ttf', 'notebooks/assets/NotoSansJP-Regular.ttf'):
    if os.path.exists(_p):
        _font_path = _p
        break
if _font_path:
    fm.fontManager.addfont(_font_path)
    plt.rcParams['font.family'] = fm.FontProperties(fname=_font_path).get_name()
else:
    print("⚠️ 日本語フォント(assets/NotoSansJP-Regular.ttf)が見つかりません。"
          "グラフの日本語表示が文字化けする可能性があります。")

In [2]:
def _read(name):
    for p in (f'../data/{name}', f'data/{name}'):
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name)

# 正式に採用が確定しているデータセットのみを読み込む。
# maria_boundaries.csv / moon_ephemeris.csv / moon_geology_grid.csv は
# 優先度3として引き続き未着手（要件定義書 Ver.1.3 第3.3節）のため、
# data/ フォルダには残しつつ、このノートブックのUIには読み込まない。
craters = _read('craters_subset.csv')
craters_3d = _read('craters_3d.csv')
deepcraters = _read('deepcraters.csv')
diviner = _read('diviner_global.csv.gz')
polar_illum = _read('lola_polar_illumination.csv')

# DeepCratersの年代インデックス（1〜5の数字）を、意味のわかる文字列にした列を追加しておく
AGE_NAMES = {
    1: '1:Pre-Nectarian(最も古い)',
    2: '2:Nectarian',
    3: '3:Imbrian',
    4: '4:Eratosthenian',
    5: '5:Copernican(最も新しい)',
}
deepcraters['Age_name'] = deepcraters['Age'].map(AGE_NAMES)

# 月面画像（背景用）の読み込み
# 出典：NASA Scientific Visualization Studio "CGI Moon Kit"（LROC WACベース、正距円筒図法、
# パブリックドメイン）。緯度経度の散布図の背景に敷いて、データ点の位置関係を直感的にする。
_moon_bg_path = None
for _p in ('assets/lroc_color_2k.jpg', 'notebooks/assets/lroc_color_2k.jpg'):
    if os.path.exists(_p):
        _moon_bg_path = _p
        break
if _moon_bg_path:
    moon_bg_img = plt.imread(_moon_bg_path)
else:
    moon_bg_img = None
    print('⚠️ 月面背景画像(assets/lroc_color_2k.jpg)が見つかりません。背景無しで表示します。')

# データセットごとに「どの列を選べるか」「日本語での説明」を定義する
# ※ Robbins Crater DBには「深さ」の情報は含まれていません（要確認事項として実データを確認した結果、
#    緯度・経度・直径に関する列のみで、深さを表す列は存在しませんでした）。深さを含む別カタログ
#    （Wang & Wu, 2021）を craters_3d として別データセットに用意しています。
# ※ LOLA日照データの permanent_shadow_fraction 列は、ここではあえて選択肢に含めていません。
#    このデータセットは「日照率のヒストグラムから自分でしきい値を決めて、永久影かどうかを
#    推理する→答え合わせをする」という2段階の探究として、この下の専用セルで扱います。
#
# color_options は {列名: {'label': 表示名, 'kind': 'categorical'（少数の区分） or 'continuous'（連続値）}}
# の形式。categoricalは凡例付きの色分け、continuousはカラーバー付きの色分けで描画する。
datasets = {
    'クレーターの直径・形（Robbins Crater DB, 直径8km以上）': {
        'df': craters,
        'columns': {
            'lat': '緯度 [度]',
            'lon': '経度 [度]',
            'diam_km': '直径 [km]',
            'diam_major_km': '長径 [km]',
            'diam_minor_km': '短径 [km]',
            'eccentricity': '離心率（真円=0に近いほど丸い）',
            'ellipticity': '扁平率（真円=1に近いほど丸い）',
            'rim_arc_fraction': 'リムが検出できた割合（0〜1）',
        },
        'color_options': {},
        'latlon': ('lon', 'lat'),
        'background': 'global',
    },
    'クレーターの直径と深さ（Wang & Wu 2021, 直径10km以上）': {
        'df': craters_3d,
        'columns': {
            'lat': '緯度 [度]',
            'lon': '経度 [度]',
            'diameter_km': '直径 [km]',
            'depth_km': '深さ [km]',
            'depth_diameter_ratio': '深さ÷直径の比',
        },
        'color_options': {
            'depth_diameter_ratio': {'label': '深さ÷直径の比（色分け）', 'kind': 'continuous'},
        },
        'latlon': ('lon', 'lat'),
        'background': 'global',
    },
    'クレーターの推定年代（DeepCraters, 直径8km以上）': {
        'df': deepcraters,
        'columns': {
            'Lat': '緯度 [度]',
            'Lon': '経度 [度]',
            'Diam_km': '直径 [km]',
            'Age': '推定年代区分（1〜5、数字が大きいほど新しい）',
        },
        'color_options': {
            'Age_name': {'label': '推定年代区分（年代名で色分け）', 'kind': 'categorical'},
            'Flags_data': {'label': 'データ取得源（CE1/CE2で色分け）', 'kind': 'categorical'},
        },
        'latlon': ('Lon', 'Lat'),
        'background': 'global',
    },
    '月面の温度（Diviner, 全球0.5度グリッド）': {
        'df': diviner,
        'columns': {
            'lon': '経度 [度]',
            'lat': '緯度 [度]',
            'temp_noon_K': '正午の温度 [K]',
            'temp_midnight_K': '深夜0時の温度 [K]',
            'temp_diff_K': '昼夜の温度差 [K]',
        },
        'color_options': {
            'temp_diff_K': {'label': '昼夜の温度差 [K]（色分け）', 'kind': 'continuous'},
        },
        'latlon': ('lon', 'lat'),
        'background': 'global',
    },
    '月の南極・北極の日照（LOLA, 約1kmグリッド）': {
        'df': polar_illum,
        'columns': {
            'lon': '経度 [度]',
            'lat': '緯度 [度]（正=北極側、負=南極側）',
            'average_illumination_percent': '平均日照率 [%]（値は目安。他の文献の数字と単純比較しないこと）',
        },
        'color_options': {
            'average_illumination_percent': {'label': '平均日照率 [%]（色分け）', 'kind': 'continuous'},
        },
        'latlon': ('lon', 'lat'),
        'background': None,  # 極域のみのデータのため、全球画像は背景に使わない（フォールバック）
    },
}

print('読み込み完了：')
for name, d in datasets.items():
    print(f' - {name}: {len(d["df"]):,} 件')

読み込み完了：
 - クレーターの直径・形（Robbins Crater DB, 直径8km以上）: 36,377 件
 - クレーターの直径と深さ（Wang & Wu 2021, 直径10km以上）: 24,982 件
 - クレーターの推定年代（DeepCraters, 直径8km以上）: 18,996 件
 - 月面の温度（Diviner, 全球0.5度グリッド）: 259,200 件
 - 月の南極・北極の日照（LOLA, 約1kmグリッド）: 157,922 件


## 探索ツール

下のプルダウンでデータセットとX軸・Y軸を選ぶと、その場で散布図と基本統計量（平均・標準偏差・相関係数）が表示されます。
点の数が多いデータセットは、見やすさのため一部だけをランダムに抜き出して表示します（「表示点数の上限」で調整可）。

緯度・経度を軸に選んだときは、背景に月面の画像が表示されます（「月面画像を背景に表示する」で切り替え可）。
「色分け」のプルダウンで、クレーターの年代や温度差などに応じて点の色を変えることもできます。

In [3]:
dataset_dropdown = widgets.Dropdown(
    options=list(datasets.keys()),
    description='データセット:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
)
x_dropdown = widgets.Dropdown(description='X軸:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
y_dropdown = widgets.Dropdown(description='Y軸:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
color_dropdown = widgets.Dropdown(description='色分け:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
sample_slider = widgets.IntSlider(
    value=3000, min=500, max=20000, step=500,
    description='表示点数の上限:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
)
bg_check = widgets.Checkbox(value=True, description='月面画像を背景に表示する（緯度経度のときのみ有効）', indent=False)
logx_check = widgets.Checkbox(value=False, description='X軸を対数目盛りにする', indent=False)
logy_check = widgets.Checkbox(value=False, description='Y軸を対数目盛りにする', indent=False)
output = widgets.Output()


def update_variable_options(change=None):
    info = datasets[dataset_dropdown.value]
    options = [(label, col) for col, label in info['columns'].items()]
    x_dropdown.options = options
    y_dropdown.options = options
    x_dropdown.value = options[0][1]
    y_dropdown.value = options[1][1] if len(options) > 1 else options[0][1]
    color_dropdown.options = [('なし', None)] + [
        (meta['label'], col) for col, meta in info['color_options'].items()
    ]
    color_dropdown.value = None
    draw_plot()


def draw_plot(change=None):
    with output:
        clear_output(wait=True)
        info = datasets[dataset_dropdown.value]
        df = info['df']
        x_col, y_col = x_dropdown.value, y_dropdown.value
        color_col = color_dropdown.value

        n = min(len(df), sample_slider.value)
        plot_df = df.sample(n=n, random_state=0) if len(df) > n else df

        fig, ax = plt.subplots(figsize=(7, 6))

        # 緯度経度の軸を選んでいて、背景表示がオンで、画像が読み込めていれば月面画像を敷く
        latlon = info.get('latlon')
        is_latlon_view = latlon and x_col == latlon[0] and y_col == latlon[1]
        if is_latlon_view and bg_check.value and info.get('background') == 'global' and moon_bg_img is not None:
            ax.imshow(moon_bg_img, extent=[-180, 180, -90, 90], aspect='auto', alpha=0.6, zorder=0)
            ax.set_xlim(-180, 180)
            ax.set_ylim(-90, 90)

        if color_col:
            kind = info['color_options'][color_col]['kind']
            if kind == 'categorical':
                categories = plot_df[color_col].astype('category')
                sc = ax.scatter(plot_df[x_col], plot_df[y_col], c=categories.cat.codes,
                                 cmap='viridis', s=8, alpha=0.75, zorder=2)
                handles, _ = sc.legend_elements()
                ax.legend(handles, categories.cat.categories, title=color_col,
                           bbox_to_anchor=(1.05, 1), loc='upper left')
            else:  # continuous
                sc = ax.scatter(plot_df[x_col], plot_df[y_col], c=plot_df[color_col],
                                 cmap='plasma', s=8, alpha=0.8, zorder=2)
                fig.colorbar(sc, ax=ax, label=info['color_options'][color_col]['label'])
        else:
            ax.scatter(plot_df[x_col], plot_df[y_col], s=8, alpha=0.5, zorder=2)

        try:
            if logx_check.value:
                ax.set_xscale('log')
            if logy_check.value:
                ax.set_yscale('log')
        except Exception:
            print('⚠️ このデータには0以下の値が含まれているため、対数グラフに変換できません。')

        ax.set_xlabel(info['columns'].get(x_col, x_col))
        ax.set_ylabel(info['columns'].get(y_col, y_col))
        ax.set_title(f"{dataset_dropdown.value}\n(表示 {n:,} / 全 {len(df):,} 件)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        if x_col != y_col:
            corr = plot_df[[x_col, y_col]].corr().iloc[0, 1]
            print(f'相関係数 r = {corr:.3f}')
        stats = plot_df[[x_col, y_col]].describe().loc[['mean', 'std', 'min', 'max']]
        display(stats)


dataset_dropdown.observe(update_variable_options, names='value')
x_dropdown.observe(draw_plot, names='value')
y_dropdown.observe(draw_plot, names='value')
color_dropdown.observe(draw_plot, names='value')
sample_slider.observe(draw_plot, names='value')
bg_check.observe(draw_plot, names='value')
logx_check.observe(draw_plot, names='value')
logy_check.observe(draw_plot, names='value')

update_variable_options()

display(widgets.VBox([
    dataset_dropdown,
    widgets.HBox([x_dropdown, y_dropdown, color_dropdown]),
    sample_slider,
    widgets.HBox([bg_check, logx_check, logy_check]),
    output,
]))

## 発展課題：月の南極・北極で「永久影」を推理する

月の極域には、太陽の光が一年中まったく当たらない「永久影（permanently shadowed region）」
と呼ばれる場所があります。水（氷）が残っている可能性があることから、将来の月面基地の
候補地として注目されています。

ここでは、あえて「ここは永久影です」という答えの列を最初から見せません。代わりに、
**「平均日照率」のヒストグラムを自分の目で見て、「これくらい低ければ永久影だろう」という
しきい値を自分で決める**→**そのしきい値で選んだ場所が、実際どれくらい永久影だったかを
確認する**、という2段階で進めます。

> ⚠️ **日照率の数値そのものについての注意**：このデータの「平均日照率」は、査読済みの
> 標準的な計算手法（Mazarico et al. 2011）によるものですが、より新しい高解像度の解析
> （Barker et al. 2021等）とは数値が異なることがあります。実際に、月で最も明るいと
> される地点の一つ（Malapert Massif）でも、他の文献では「約90%」とされる一方、この
> データでは「約40%」ほどしか出ません。**「暗いか明るいか」という順序・傾向は信頼できますが、
> 「◯◯%」という絶対値を、他の資料の数字とそのまま比べないようにしてください。**

In [4]:
illum_values = polar_illum['average_illumination_percent']

threshold_slider = widgets.FloatSlider(
    value=15.0, min=0.0, max=float(int(illum_values.max()) + 1), step=0.5,
    description='しきい値 [%]:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px'),
)
reveal_button = widgets.Button(description='答え合わせをする', button_style='info')
hist_output = widgets.Output()
reveal_output = widgets.Output()


def draw_histogram(change=None):
    with hist_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(illum_values, bins=40, color='steelblue', alpha=0.75)
        ax.axvline(threshold_slider.value, color='crimson', linestyle='--',
                   label=f'しきい値 {threshold_slider.value:.1f}%')
        ax.set_xlabel('平均日照率 [%]')
        ax.set_ylabel('地点数')
        ax.set_title('月の南極・北極：平均日照率のヒストグラム')
        ax.legend()
        plt.tight_layout()
        plt.show()
        print('このヒストグラムを見て、「これくらい低ければ永久影だろう」と思うしきい値をスライダーで選んでみましょう。')
        print('決まったら「答え合わせをする」を押してください。')


def reveal_answer(_):
    with reveal_output:
        clear_output(wait=True)
        selected = polar_illum[polar_illum['average_illumination_percent'] <= threshold_slider.value]
        n = len(selected)
        if n == 0:
            print('このしきい値では該当する地点がありませんでした。しきい値を上げて試してみましょう。')
            return
        # 答え合わせに使う指標：
        #  - zero_rate: 選んだ地点のうち「永久影率が完全にゼロ(=まったく永久影ではない)」だった割合
        #    （しきい値が緩すぎたことに気づくための「外れ」の割合）
        #  - median_shadow: 選んだ地点の永久影率の中央値（1.000に近いほど、選んだ集団全体が
        #    永久影に近いと言える）
        zero_rate = (selected['permanent_shadow_fraction'] == 0).mean()
        median_shadow = selected['permanent_shadow_fraction'].median()
        print(f'しきい値 {threshold_slider.value:.1f}% 以下に該当する地点：{n:,}件')
        print(f'そのうち、実は「まったく永久影ではなかった」地点の割合：{zero_rate * 100:.1f}%')
        print(f'選んだ地点全体の永久影率（中央値）：{median_shadow:.3f}（1.000に近いほど確実に永久影）')
        if zero_rate <= 0.05:
            print('→ かなり良い線です。このしきい値で選んだ場所は、ほぼ確実に永久影と言えます。')
        elif zero_rate <= 0.20:
            print('→ おおむね永久影と言えそうですが、まったく違う場所もそれなりに混じっています。')
            print('   しきい値をもう少し厳しく（小さく）してみるとどうなるか試してみましょう。')
        else:
            print('→ このしきい値だと、「思ったより永久影じゃない場所」がかなり混じっています。')
            print('   しきい値を下げてみると、この割合がどう変わるか確認してみましょう。')

        # 答え合わせ専用のマップ：選んだ地点を、実際の永久影率(permanent_shadow_fraction)で
        # 色分けして表示する。この列は答え合わせのこの場面でのみ使う。
        n_show = min(n, 5000)
        show_df = selected.sample(n=n_show, random_state=0) if n > n_show else selected
        fig, ax = plt.subplots(figsize=(7, 5))
        sc = ax.scatter(show_df['lon'], show_df['lat'], c=show_df['permanent_shadow_fraction'],
                         cmap='cividis_r', s=10, alpha=0.85)
        fig.colorbar(sc, ax=ax, label='永久影率（答え合わせ用）')
        ax.set_xlabel('経度 [度]')
        ax.set_ylabel('緯度 [度]（正=北極側、負=南極側）')
        ax.set_title(f'しきい値{threshold_slider.value:.1f}%以下の地点の、実際の永久影率')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()


threshold_slider.observe(draw_histogram, names='value')
reveal_button.on_click(reveal_answer)

draw_histogram()
display(widgets.VBox([threshold_slider, hist_output, reveal_button, reveal_output]))

## 問いのヒント（迷ったときに）

正解ではなく、あくまで出発点の例です。自分で見つけた組み合わせを優先してください。

1. クレーターの直径と深さの関係（「クレーターの直径と深さ（Wang & Wu 2021）」データセットを使う）
2. クレーターの直径と、離心率・扁平率（＝どれくらい丸いか）の関係
3. クレーターの緯度・経度分布のかたより（密集している場所とそうでない場所。背景の月面画像と見比べてみる）
4. 緯度と正午の温度の関係
5. 同じ場所での「正午の温度」と「深夜0時の温度」の差（昼夜温度差）と、緯度との関係（「色分け：昼夜の温度差」も使ってみる）
6. クレーターの推定年代（Age）と、直径や分布との関係（DeepCratersのデータのみで完結。「色分け：推定年代区分」が便利）
7. クレーターの直径と深さの比（depth_diameter_ratio）は、場所によって違いがあるか（「色分け：深さ÷直径の比」を使う）
8. 月の南極・北極の「永久影」を推理する（上の「発展課題」の専用ツールを使う）

> **注記**：当初の教材案が参照していたRobbins Crater Database (2018) には、実際に確認したところ
> **深さ（Depth）を表す列は含まれていません**（緯度・経度・直径・形状に関する列のみ）。
> そのため「直径と深さ」を調べたい場合は、深さを含む別カタログ（Wang & Wu, 2021）の
> データセットを使ってください。

## データの出典

- Robbins, S. J. (2018). *A New Global Database of Lunar Impact Craters >1–2 km*. USGS Astrogeology Science Center.
  https://astrogeology.usgs.gov/search/map/Moon/Research/Craters/lunar_crater_database_robbins_2018
- Wang, Y., Wu, B. (2021). *An improved global catalog of lunar impact craters (≥1 km) with 3D
  morphometric information*. JGR Planets, 126, e2020JE006728. Zenodo:
  https://doi.org/10.5281/zenodo.4983248 （クレーターの深さを含むカタログ、CC BY 4.0）
- Yang, C., Guan, R. (2020). *CE_DeepCraters* (Aged Lunar Crater Database). figshare.
  https://doi.org/10.6084/m9.figshare.12768539
- Williams, J.-P. et al. (2017). *The global surface temperatures of the Moon as measured by the
  Diviner Lunar Radiometer Experiment*. Icarus, 283, 300-325. データ配布：
  https://www.diviner.ucla.edu/data （UCLA Diviner Lunar Radiometer Experiment チーム提供）。
  本教材では、subsolar経度を15度刻みで撮った瞬間温度マップ24枚（0.5度グリッド）を
  現地時間に位相合わせし、地点ごとに現地時間0〜23時の温度カーブ（t_lt00〜t_lt23）に
  作り替えている。日変化カーブが意味を持つのは概ね|緯度|<70度（極付近は太陽が地平線
  近くを回るためカーブが平坦・不規則になる）。
- Mazarico, E. et al. (2011). *Illumination conditions of the lunar polar regions using LOLA
  topography*. Icarus, 211, 1066-1081. データ配布：LRO LOLA Team (NASA GSFC), PDS Geosciences
  Node（南極・北極の平均日照率・永久影マップ、約1kmグリッドに再集計）
- NASA Scientific Visualization Studio. *CGI Moon Kit*（LROC WACベースの月面色調モザイク、
  正距円筒図法、パブリックドメイン）。散布図の背景画像として使用。
  https://svs.gsfc.nasa.gov/4720

> `maria_boundaries.csv`（USGS地名辞典）・`moon_ephemeris.csv`（JPL HORIZONS）・
> `moon_geology_grid.csv`（USGS統合地質図）は、`data/`フォルダには用意されていますが、
> 2026-08-28時点でこのノートブックのUIには含まれていません（優先度3として検証・採否が
> 未確定のため）。詳細は`docs/requirements_v1.3.md`を参照してください。